# 03 · Per-task AIM improvement over LS

**Which individual properties actually benefit from AIM?**

    improvement_t = 100 · (MAE_LS,t − MAE_AIM,t) / MAE_LS,t

Bars right of zero mean AIM beat plain linear scalarization on that property.
LS is the reference rather than STL on purpose: it is the do-nothing multi-task
baseline — same architecture, same budget, no gradient surgery — so the
difference isolates the intervention itself.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, ".")
import numpy as np, matplotlib.pyplot as plt
import common as C
C.apply_style()
print("results trees:")
for b, d in C.RESULTS.items():
    for tag, p in d.items():
        print(f"  {b:9s} {tag:7s} {'OK ' if p.is_dir() else 'MISSING'} {p}")

In [ ]:
TAG = "11task"
AIM = "AIM-Matrix"       # or "AIM-Scalar"
ORDER = "gain"           # "gain" sorts by mean improvement; "task" keeps QM9 order

data, tasks = {}, None
for b in ("Uni-Mol", "GNN"):
    d = C.load_runs(b, TAG)
    ls, aim = d["mtl"]["LS"], d["mtl"][AIM]
    tasks = tasks or d["tasks"]
    data[b] = {t: 100 * (ls[t] - aim[t]) / ls[t] for t in d["tasks"]}
    wins = [t for t, v in data[b].items() if v > 0]
    print(f"{b:9s} AIM better on {len(wins)}/{len(data[b])}: {', '.join(wins) or 'none'}")

In [ ]:
if ORDER == "gain":
    tasks = sorted(tasks, key=lambda t: np.mean([data[b][t] for b in data]))

fig, ax = plt.subplots(figsize=(11, max(3.2, .42 * len(tasks) + 2.3)))
y = np.arange(len(tasks)); h = .38
for s, b in enumerate(data):
    vals = [data[b][t] for t in tasks]
    ax.barh(y + (s - .5) * h, vals, h * .92, color=C.BACKBONE_COLOUR[b],
            edgecolor=C.SURFACE, linewidth=1.5, zorder=3, label=b)
ax.axvline(0, color=C.AXIS, lw=1.2, zorder=4)
ax.set_yticks(y, tasks); ax.invert_yaxis(); ax.margins(x=.16)
ax.set_xlabel("AIM better than LS  (%)   →")
for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
ax.grid(True, axis="x", zorder=0); ax.set_axisbelow(True)
fig.suptitle("Per-task AIM improvement over linear scalarization", x=.012, y=.99,
             ha="left", fontsize=15, fontweight="bold", color=C.INK)
fig.text(.012, .93, f"{len(tasks)} QM9 tasks · {AIM} vs LS · Uni-Mol (pretrained) "
         f"vs GNN (from scratch) · n_train={C.N_TRAIN}, seed {C.SEED}",
         fontsize=9.5, color=C.MUTED, ha="left")
ax.legend(loc="lower right")
fig.subplots_adjust(left=.12, right=.97, top=.84, bottom=.13)
C.save(fig, f"aim_vs_ls_{TAG}", TAG); plt.show()